In [1]:
# Vérification de CUDA et PyTorch
import torch
import sys

print("=== VÉRIFICATION DE L'ENVIRONNEMENT CUDA/PYTORCH ===")
print(f"Version Python: {sys.version}")
print(f"Version PyTorch: {torch.__version__}")

# Vérification de CUDA
print(f"\nCUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nombre de GPU(s): {torch.cuda.device_count()}")
    print(f"GPU actuel: {torch.cuda.current_device()}")
    print(f"Nom du GPU: {torch.cuda.get_device_name(0)}")
    print(f"Version CUDA (PyTorch): {torch.version.cuda}")
    print(f"Mémoire GPU totale: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"Mémoire GPU allouée: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Mémoire GPU libre: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    
    # Test simple pour vérifier que CUDA fonctionne
    print("\n=== TEST CUDA ===")
    device = torch.device("cuda:0")
    x = torch.randn(3, 3).to(device)
    y = torch.randn(3, 3).to(device)
    z = torch.matmul(x, y)
    print(f"Test de multiplication matricielle sur GPU: {'✓ SUCCÈS' if z.is_cuda else '✗ ÉCHEC'}")
    print(f"Résultat sur device: {z.device}")
else:
    print("CUDA n'est pas disponible. Vérifiez l'installation de PyTorch avec support CUDA.")

print("\n" + "="*60)

=== VÉRIFICATION DE L'ENVIRONNEMENT CUDA/PYTORCH ===
Version Python: 3.11.0rc1 (main, Aug 12 2022, 10:02:14) [GCC 11.2.0]
Version PyTorch: 2.8.0+cu128

CUDA disponible: True
Nombre de GPU(s): 1
GPU actuel: 0
Nom du GPU: NVIDIA GeForce RTX 4090
Version CUDA (PyTorch): 12.8
Mémoire GPU totale: 24.0 GB
Mémoire GPU allouée: 0.00 GB
Mémoire GPU libre: 0.00 GB

=== TEST CUDA ===
Test de multiplication matricielle sur GPU: ✓ SUCCÈS
Résultat sur device: cuda:0



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import librosa
import IPython.display as ipd
from collections import defaultdict

# Configuration pour les graphiques
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 8)

ModuleNotFoundError: No module named 'librosa'

In [ ]:
from datasets import load_dataset
ds = load_dataset("diarizers-community/voxconverse")

print(ds)


In [ ]:
# Explorer la structure du dataset VoxConverse
print("=== STRUCTURE DU DATASET VOXCONVERSE ===")
print(f"Splits disponibles: {list(ds.keys())}")
print(f"Nombre d'éléments dans dev: {len(ds['dev'])}")
print(f"Nombre d'éléments dans test: {len(ds['test'])}")

print("\n=== FEATURES DISPONIBLES ===")
print("Features:", ds['dev'].features)

print("\n=== PREMIER ÉLÉMENT DU DATASET (DEV) ===")
first_item = ds['dev'][0]
for key, value in first_item.items():
    if key == 'audio':
        print(f"{key}: array de shape {np.array(value['array']).shape}, sample_rate={value['sampling_rate']}")
    elif key == 'timestamps_start':
        print(f"{key}: {len(value)} timestamps de début - premiers: {value[:5] if len(value) > 5 else value}")
    elif key == 'timestamps_end':
        print(f"{key}: {len(value)} timestamps de fin - premiers: {value[:5] if len(value) > 5 else value}")
    elif key == 'speakers':
        print(f"{key}: {len(value)} labels de locuteurs - premiers: {value[:10] if len(value) > 10 else value}")
        unique_speakers = list(set(value))
        print(f"  Locuteurs uniques: {unique_speakers}")
        print(f"  Nombre de locuteurs uniques: {len(unique_speakers)}")
    elif isinstance(value, str):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value}")

# Analyser la structure des timestamps et speakers
if 'timestamps_start' in first_item and 'timestamps_end' in first_item and 'speakers' in first_item:
    timestamps_start = first_item['timestamps_start']
    timestamps_end = first_item['timestamps_end']
    speakers = first_item['speakers']
    
    print(f"\n=== ANALYSE DES SEGMENTS DE DIARIZATION ===")
    print(f"Nombre de segments: {len(timestamps_start)}")
    
    if len(timestamps_start) > 0:
        print(f"Premiers segments:")
        for i in range(min(5, len(timestamps_start))):
            duration = timestamps_end[i] - timestamps_start[i]
            print(f"  Segment {i+1}: {timestamps_start[i]:.2f}s-{timestamps_end[i]:.2f}s ({duration:.2f}s), Locuteur: {speakers[i]}")

print(f"\nDurée audio: {len(first_item['audio']['array']) / first_item['audio']['sampling_rate']:.2f} secondes")

In [ ]:
def analyze_voxconverse_item(item):
    """
    Analyse un élément du dataset VoxConverse et extrait les informations de diarization
    """
    # Créer les segments à partir des timestamps
    segments = []
    timestamps_start = item['timestamps_start']
    timestamps_end = item['timestamps_end'] 
    speakers = item['speakers']
    
    for i in range(len(timestamps_start)):
        segment_info = {
            'start': timestamps_start[i],
            'end': timestamps_end[i],
            'speaker': speakers[i]
        }
        segments.append(segment_info)
    
    # Trier par temps de début
    segments.sort(key=lambda x: x['start'])
    
    item_info = {
        'audio_id': f"voxconverse_item",  # VoxConverse n'a pas d'audio_id explicite
        'audio': item['audio'],
        'segments': segments
    }
    
    return item_info

def plot_voxconverse_timeline(item_info, max_time=None, title_suffix=""):
    """
    Trace la timeline des segments de locuteurs pour un élément VoxConverse
    """
    segments = item_info['segments']
    audio_id = item_info['audio_id']
    
    if not segments:
        print(f"Aucun segment trouvé pour {audio_id}")
        return []
    
    # Déterminer la durée maximale
    if max_time is None:
        max_time = max([seg['end'] for seg in segments]) if segments else 60
    
    fig, ax = plt.subplots(1, 1, figsize=(15, 6))
    
    # Timeline des locuteurs
    speakers = list(set(seg['speaker'] for seg in segments))
    speaker_colors = plt.cm.Set3(np.linspace(0, 1, len(speakers)))
    speaker_color_map = dict(zip(speakers, speaker_colors))
    
    segments_in_range = []
    
    for i, segment in enumerate(segments):
        start_time = segment['start']
        end_time = min(segment['end'], max_time)
        speaker = segment['speaker']
        
        if start_time < max_time:  # Seulement si le segment commence dans la période d'intérêt
            segments_in_range.append(segment)
            
            # Tracer le segment
            ax.barh(speaker, end_time - start_time, left=start_time, 
                    color=speaker_color_map[speaker], alpha=0.7, 
                    edgecolor='black', linewidth=0.5)
            
            # Ajouter le numéro du segment si assez long
            if end_time - start_time > 2:  
                ax.text(start_time + (end_time - start_time)/2, speaker, 
                        f"{i+1}", ha='center', va='center', fontsize=8, fontweight='bold')
    
    ax.set_xlabel('Temps (secondes)')
    ax.set_ylabel('Locuteur')
    ax.set_title(f'Timeline des locuteurs - VoxConverse{title_suffix}')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, max_time)
    
    plt.tight_layout()
    plt.show()
    
    # Afficher les statistiques
    print(f"\n=== STATISTIQUES ===")
    print(f"Période analysée: 0-{max_time}s")
    print(f"Nombre de segments dans cette période: {len(segments_in_range)}")
    print(f"Nombre total de segments: {len(segments)}")
    print(f"Nombre de locuteurs: {len(speakers)}")
    print(f"Locuteurs: {speakers}")
    
    if segments_in_range:
        durations = [min(s['end'], max_time) - s['start'] for s in segments_in_range if s['start'] < max_time]
        print(f"Durée moyenne des segments: {np.mean(durations):.2f} secondes")
        
        # Temps de parole par locuteur
        speaker_time = {}
        for seg in segments_in_range:
            speaker = seg['speaker']
            duration = min(seg['end'], max_time) - seg['start']
            if duration > 0:
                speaker_time[speaker] = speaker_time.get(speaker, 0) + duration
        
        print(f"\nTemps de parole par locuteur:")
        for speaker, time in sorted(speaker_time.items()):
            percentage = (time / max_time) * 100
            print(f"  {speaker}: {time:.2f}s ({percentage:.1f}%)")
    
    return segments_in_range

def get_voxconverse_audio_extract(item, start_time=0, duration=60):
    """
    Extrait une portion d'audio d'un élément VoxConverse
    """
    audio_array = np.array(item['audio']['array'])
    sample_rate = item['audio']['sampling_rate']
    
    # Calculer les indices d'échantillons
    start_sample = int(start_time * sample_rate)
    end_sample = int((start_time + duration) * sample_rate)
    
    # S'assurer de ne pas dépasser la longueur de l'audio
    start_sample = min(start_sample, len(audio_array))
    end_sample = min(end_sample, len(audio_array))
    
    # Extraire la portion d'audio
    if start_sample < len(audio_array):
        extracted_audio = audio_array[start_sample:end_sample]
        actual_duration = len(extracted_audio) / sample_rate
    else:
        extracted_audio = np.array([])
        actual_duration = 0
    
    total_duration = len(audio_array) / sample_rate
    
    print(f"Audio extrait: {actual_duration:.2f} secondes (de {start_time:.2f}s à {start_time + actual_duration:.2f}s)")
    print(f"Durée totale de l'audio: {total_duration:.2f} secondes")
    
    return extracted_audio, sample_rate

In [ ]:
# Explorer les premiers éléments du dataset VoxConverse
print("=== EXPLORATION DES ÉLÉMENTS VOXCONVERSE ===")

# Utiliser le split 'dev' puisqu'il n'y a pas de 'train'
split_to_use = 'dev'
dataset_split = ds[split_to_use]

# Analyser les premiers éléments
num_items_to_explore = min(3, len(dataset_split))
print(f"Exploration des {num_items_to_explore} premiers éléments du split '{split_to_use}'")

for i in range(num_items_to_explore):
    item = dataset_split[i]
    
    print(f"\n{'='*60}")
    print(f"ÉLÉMENT {i+1}/{num_items_to_explore}")
    print(f"{'='*60}")
    
    # Analyser l'élément
    item_info = analyze_voxconverse_item(item)
    
    print(f"Durée audio totale: {len(item['audio']['array']) / item['audio']['sampling_rate']:.2f} secondes")
    print(f"Sample rate: {item['audio']['sampling_rate']} Hz")
    print(f"Nombre de segments: {len(item_info['segments'])}")
    
    if item_info['segments']:
        # Afficher les speakers uniques
        unique_speakers = list(set(seg['speaker'] for seg in item_info['segments']))
        print(f"Locuteurs uniques: {unique_speakers} ({len(unique_speakers)} locuteurs)")
        
        # Afficher les premiers segments
        print(f"\nPremiers segments:")
        for j, seg in enumerate(item_info['segments'][:8]):  # Afficher jusqu'à 8 segments
            duration = seg['end'] - seg['start']
            print(f"  Segment {j+1}: {seg['start']:.2f}s-{seg['end']:.2f}s ({duration:.2f}s), Locuteur: {seg['speaker']}")
        
        if len(item_info['segments']) > 8:
            print(f"  ... et {len(item_info['segments']) - 8} autres segments")
            
        # Calculer les statistiques de base
        durations = [seg['end'] - seg['start'] for seg in item_info['segments']]
        total_speech_time = sum(durations)
        total_audio_time = len(item['audio']['array']) / item['audio']['sampling_rate']
        coverage = (total_speech_time / total_audio_time) * 100
        
        print(f"\nStatistiques:")
        print(f"  Temps total de parole: {total_speech_time:.2f}s")
        print(f"  Couverture de l'audio: {coverage:.1f}%")
        print(f"  Durée moyenne des segments: {np.mean(durations):.2f}s")
    
    print(f"\n" + "-"*60)

In [ ]:
# Analyse détaillée du premier élément VoxConverse avec timeline et audio
print("=== ANALYSE DÉTAILLÉE DU PREMIER ÉLÉMENT VOXCONVERSE ===")

if len(ds['dev']) > 0:
    first_item = ds['dev'][0]
    
    # Analyser l'élément
    item_info = analyze_voxconverse_item(first_item)
    
    total_duration = len(first_item['audio']['array']) / first_item['audio']['sampling_rate']
    print(f"Durée totale: {total_duration:.2f} secondes")
    print(f"Nombre de segments: {len(item_info['segments'])}")
    
    # Tracer la timeline pour les 2 premières minutes
    if item_info['segments']:
        print(f"\n=== TIMELINE DES LOCUTEURS (2 premières minutes) ===")
        segments_120s = plot_voxconverse_timeline(item_info, max_time=120, title_suffix=" (2 premières minutes)")
        
        # Afficher les détails des segments dans les 2 premières minutes
        print(f"\n=== DÉTAIL DES SEGMENTS (2 premières minutes) ===")
        for j, seg in enumerate(segments_120s[:12]):  # Limiter à 12 segments
            duration = min(seg['end'], 120) - seg['start']
            print(f"Segment {j+1}: {seg['start']:.2f}s-{seg['end']:.2f}s ({duration:.2f}s), "
                  f"Locuteur: {seg['speaker']}")
        
        if len(segments_120s) > 12:
            print(f"... et {len(segments_120s) - 12} autres segments")
        
        # Extraire et jouer l'audio des 2 premières minutes (120 secondes)
        print(f"\n=== AUDIO DES 2 PREMIÈRES MINUTES ===")
        two_minutes_audio, sr = get_voxconverse_audio_extract(first_item, start_time=0, duration=120)
        
        if len(two_minutes_audio) > 0:
            print(f"Sample rate: {sr} Hz")
            print(f"Lecteur audio des 2 premières minutes:")
            display(ipd.Audio(two_minutes_audio, rate=sr))
        else:
            print("Aucun audio disponible pour les 2 premières minutes")
    else:
        print("Aucun segment de diarization trouvé")
else:
    print("Dataset vide")

In [ ]:
# Analyse de la distribution du temps des conversations dans VoxConverse
print("=== DISTRIBUTION DU TEMPS DES CONVERSATIONS - VOXCONVERSE ===")

# Analyser tous les éléments du dataset
all_durations = []
all_segments_count = []
all_speakers_count = []

# Analyser le split 'dev'
print("Analyse du split 'dev'...")
for i, item in enumerate(ds['dev']):
    if i % 50 == 0:
        print(f"  Traitement élément {i+1}/{len(ds['dev'])}")
    
    # Calculer la durée totale de l'audio
    duration = len(item['audio']['array']) / item['audio']['sampling_rate']
    all_durations.append(duration)
    
    # Compter les segments
    num_segments = len(item['timestamps_start'])
    all_segments_count.append(num_segments)
    
    # Compter les locuteurs uniques
    unique_speakers = len(set(item['speakers']))
    all_speakers_count.append(unique_speakers)

# Analyser le split 'test'
print("Analyse du split 'test'...")
for i, item in enumerate(ds['test']):
    if i % 50 == 0:
        print(f"  Traitement élément {i+1}/{len(ds['test'])}")
    
    # Calculer la durée totale de l'audio
    duration = len(item['audio']['array']) / item['audio']['sampling_rate']
    all_durations.append(duration)
    
    # Compter les segments
    num_segments = len(item['timestamps_start'])
    all_segments_count.append(num_segments)
    
    # Compter les locuteurs uniques
    unique_speakers = len(set(item['speakers']))
    all_speakers_count.append(unique_speakers)

# Convertir en numpy arrays pour les calculs
all_durations = np.array(all_durations)
all_segments_count = np.array(all_segments_count)
all_speakers_count = np.array(all_speakers_count)

print(f"\nNombre total de conversations analysées: {len(all_durations)}")
print(f"  - Split dev: {len(ds['dev'])} conversations")
print(f"  - Split test: {len(ds['test'])} conversations")

# Statistiques générales
print(f"\n=== STATISTIQUES GÉNÉRALES ===")
print(f"Durée totale du dataset: {all_durations.sum()/3600:.2f} heures")
print(f"Durée moyenne par conversation: {all_durations.mean():.2f} secondes ({all_durations.mean()/60:.2f} minutes)")
print(f"Durée médiane: {np.median(all_durations):.2f} secondes")
print(f"Durée minimale: {all_durations.min():.2f} secondes")
print(f"Durée maximale: {all_durations.max():.2f} secondes")
print(f"Écart-type: {all_durations.std():.2f} secondes")

print(f"\nSegments de diarization:")
print(f"Nombre moyen de segments par conversation: {all_segments_count.mean():.1f}")
print(f"Nombre médian de segments: {np.median(all_segments_count):.0f}")
print(f"Min/Max segments: {all_segments_count.min()} / {all_segments_count.max()}")

print(f"\nNombre de locuteurs:")
print(f"Nombre moyen de locuteurs par conversation: {all_speakers_count.mean():.1f}")
print(f"Nombre médian de locuteurs: {np.median(all_speakers_count):.0f}")
print(f"Min/Max locuteurs: {all_speakers_count.min()} / {all_speakers_count.max()}")

# Créer les graphiques de distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distribution des durées
axes[0, 0].hist(all_durations/60, bins=30, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Durée (minutes)')
axes[0, 0].set_ylabel('Nombre de conversations')
axes[0, 0].set_title('Distribution des durées des conversations')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(all_durations.mean()/60, color='red', linestyle='--', label=f'Moyenne: {all_durations.mean()/60:.1f}min')
axes[0, 0].axvline(np.median(all_durations)/60, color='orange', linestyle='--', label=f'Médiane: {np.median(all_durations)/60:.1f}min')
axes[0, 0].legend()

# Distribution des nombres de segments
axes[0, 1].hist(all_segments_count, bins=20, alpha=0.7, edgecolor='black', color='green')
axes[0, 1].set_xlabel('Nombre de segments')
axes[0, 1].set_ylabel('Nombre de conversations')
axes[0, 1].set_title('Distribution du nombre de segments')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(all_segments_count.mean(), color='red', linestyle='--', label=f'Moyenne: {all_segments_count.mean():.1f}')
axes[0, 1].legend()

# Distribution des nombres de locuteurs
axes[1, 0].hist(all_speakers_count, bins=range(int(all_speakers_count.min()), int(all_speakers_count.max())+2), 
                alpha=0.7, edgecolor='black', color='purple')
axes[1, 0].set_xlabel('Nombre de locuteurs')
axes[1, 0].set_ylabel('Nombre de conversations')
axes[1, 0].set_title('Distribution du nombre de locuteurs')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticks(range(int(all_speakers_count.min()), int(all_speakers_count.max())+1))

# Relation durée vs nombre de segments
axes[1, 1].scatter(all_durations/60, all_segments_count, alpha=0.6, s=20)
axes[1, 1].set_xlabel('Durée (minutes)')
axes[1, 1].set_ylabel('Nombre de segments')
axes[1, 1].set_title('Relation durée vs nombre de segments')
axes[1, 1].grid(True, alpha=0.3)

# Calculer la corrélation
correlation = np.corrcoef(all_durations, all_segments_count)[0, 1]
axes[1, 1].text(0.05, 0.95, f'Corrélation: {correlation:.3f}', 
                transform=axes[1, 1].transAxes, bbox=dict(boxstyle="round", facecolor='wheat'))

plt.tight_layout()
plt.show()

# Percentiles et quartiles
print(f"\n=== PERCENTILES DES DURÉES ===")
percentiles = [10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    duration_p = np.percentile(all_durations, p)
    print(f"P{p}: {duration_p:.2f}s ({duration_p/60:.2f}min)")

# Répartition par catégories de durée
print(f"\n=== RÉPARTITION PAR CATÉGORIES DE DURÉE ===")
short_conv = np.sum(all_durations < 60)  # < 1 minute
medium_conv = np.sum((all_durations >= 60) & (all_durations < 300))  # 1-5 minutes
long_conv = np.sum((all_durations >= 300) & (all_durations < 600))  # 5-10 minutes
very_long_conv = np.sum(all_durations >= 600)  # > 10 minutes

print(f"Conversations courtes (< 1min): {short_conv} ({short_conv/len(all_durations)*100:.1f}%)")
print(f"Conversations moyennes (1-5min): {medium_conv} ({medium_conv/len(all_durations)*100:.1f}%)")
print(f"Conversations longues (5-10min): {long_conv} ({long_conv/len(all_durations)*100:.1f}%)")
print(f"Conversations très longues (> 10min): {very_long_conv} ({very_long_conv/len(all_durations)*100:.1f}%)")